# Auditoria SIH/SUS: reextração N18, versões de AIH e comparação com TabNet

Reextrai os DBCs locais com diagnóstico principal N18 e data de internação em 2024–2025, confronta o CSV preliminar, descreve mudanças entre competências e testa regras de contagem contra o TabNet N17–N19. Os arquivos brutos são preservados. O notebook não escolhe deduplicação apenas por proximidade numérica.


## Fonte e interpretação

A nota técnica do TabNet distingue processamento de atendimento. “Internações” conta AIHs aprovadas sem prorrogações por longa permanência; transferências e reinternações continuam contadas. Fonte: [Notas técnicas SIH/DATASUS](https://tabnet.datasus.gov.br/cgi/sih/Proced_hosp_loc_int_2008.pdf).

A documentação SIH descreve IDENT=1 como AIH inicial e IDENT=5 como continuidade renovável em longa permanência. Fonte: [Wiki SIH/SUS](https://wiki.saude.gov.br/sih/index.php/P%C3%A1gina_principal). O filtro IDENT será sensibilidade, não presumido como implementação exata do TabNet.

A tabela TabNet é N17–N19 e o microdado é N18 do diagnóstico principal; a comparação requer também competência, local de internação, atualização e unidade compatíveis. N_AIH identifica autorização, não pessoa. “Última versão” significa apenas a competência máxima local (dez/2025), pois pode haver revisões posteriores. MORTE é óbito durante a internação, não sobrevida nem efeito terapêutico.


## Preparar ambiente

Os wheels verificados estão em `Dados/04_documentacao_e_auditoria/pacotes_dbc/`. Execute a célula seguinte para instalar `dbc_reader` e `dbf-reader` no ambiente do kernel, sem rede; depois reinicie o kernel. O leitor percorre DBC diretamente e não cria DBF persistente. Em Apple Silicon, a distribuição inclui um binário Darwin x86_64 que é executado via Rosetta; o código corrige a permissão de execução somente no ambiente Python.

O script `Dados/04_documentacao_e_auditoria/auditar_sih_dbc_reconciliacao.py` contém a reextração integral já validada e gera saídas em pasta exclusiva, sem sobrescrever os CSVs atuais.


In [ ]:
from pathlib import Path
import subprocess, sys

candidatos = (Path.cwd(), *Path.cwd().parents)
raiz_projeto = next(p for p in candidatos if (p / "Trabalho-final/Notebook").is_dir())
wheels = raiz_projeto / "Trabalho-final/Dados/04_documentacao_e_auditoria/pacotes_dbc"
subprocess.check_call([
    sys.executable, "-m", "pip", "install", "--no-index", "--find-links", str(wheels),
    "dbc_reader==0.1.2", "dbf-reader==0.2.2",
])


In [ ]:
from pathlib import Path
from collections import Counter, defaultdict
from datetime import datetime
from decimal import Decimal, InvalidOperation
from tempfile import TemporaryDirectory
import csv, hashlib, re, sys, time, traceback, stat

import dbc_reader
from dbc_reader import DbcReader

# O wheel PyPI contém o binário Darwin sem bit de execução; habilite-o apenas no venv.
if sys.platform == "darwin":
    blast = Path(dbc_reader.__file__).parent / "bin" / "blast-dbf-darwin"
    if blast.is_file() and not (blast.stat().st_mode & stat.S_IXUSR):
        blast.chmod(blast.stat().st_mode | stat.S_IXUSR)

print("Python:", sys.version.split()[0], "| DBC: dbc_reader 0.1.2")


In [ ]:
def localizar_raiz():
    for atual in (Path.cwd(), *Path.cwd().parents):
        if (atual / "Dados/01_fontes_originais/SIH/SIH_RD_2024_2025_bruto").is_dir():
            return atual
        if (atual / "Dados/SIH_RD_2024_2025_bruto").is_dir():
            return atual
        if (atual / "Trabalho-final/Dados/01_fontes_originais/SIH/SIH_RD_2024_2025_bruto").is_dir():
            return atual / "Trabalho-final"
        if (atual / "Trabalho-final/Dados/SIH_RD_2024_2025_bruto").is_dir():
            return atual / "Trabalho-final"
    raise FileNotFoundError("Pasta local SIH_RD_2024_2025_bruto não encontrada.")

TRABALHO = localizar_raiz()
DADOS = TRABALHO / "Dados"
DBC_CANDIDATOS = [
    DADOS / "01_fontes_originais/SIH/SIH_RD_2024_2025_bruto",
    DADOS / "SIH_RD_2024_2025_bruto",
]
DBC_DIR = next((p for p in DBC_CANDIDATOS if p.is_dir()), None)
if DBC_DIR is None:
    raise FileNotFoundError("Diretório DBC SIH não encontrado.")
DERIVADOS = DADOS / "02_derivados_preliminares/SIH"
if not DERIVADOS.is_dir():
    DERIVADOS = DADOS
CSV_N18 = DERIVADOS / "sih_n18_microdados_uf_ano_mes_2024_2025.csv"
CSV_TABNET = DERIVADOS / "sih_insuficiencia_renal_n17_n19_uf_mes_2024_2025_analitico.csv"
MANIFESTO = DBC_DIR / "SHA256SUMS.txt"

PADRAO = re.compile(r"^RD(?P<uf>[A-Z]{2})(?P<aa>\d{2})(?P<mm>\d{2})$", re.I)
def meta_arquivo(path):
    m = PADRAO.match(path.stem)
    if not m: return None
    ano, mes = 2000 + int(m["aa"]), int(m["mm"])
    if mes not in range(1, 13): raise ValueError(path.name)
    return {"path": path, "uf_arq": m["uf"].upper(), "comp": f"{ano}{mes:02d}"}

arquivos = sorted(
    (x for p in DBC_DIR.glob("*.dbc") if (x := meta_arquivo(p))),
    key=lambda x: (x["comp"], x["uf_arq"]),
)
assert len(arquivos) == 648
assert len({(x["uf_arq"], x["comp"]) for x in arquivos}) == 648
assert len({x["uf_arq"] for x in arquivos}) == 27
assert CSV_N18.is_file() and CSV_TABNET.is_file()
print("DBC:", len(arquivos), "| bytes:", sum(x["path"].stat().st_size for x in arquivos))

def conferir_hashes():
    saida = []
    for linha in MANIFESTO.read_text(encoding="utf-8").splitlines():
        if not linha.strip(): continue
        esperado, nome = linha.split(maxsplit=1)
        path = MANIFESTO.parent / nome.strip().lstrip("* ")
        if not path.is_file():
            saida.append((nome, "ausente")); continue
        h = hashlib.sha256()
        with path.open("rb") as f:
            for bloco in iter(lambda: f.read(1024*1024), b""): h.update(bloco)
        saida.append((nome, "ok" if h.hexdigest() == esperado.lower() else "divergente"))
    return saida

hashes = conferir_hashes()
print("Manifesto:", Counter(status for _, status in hashes))
assert all(status == "ok" for _, status in hashes)


In [ ]:
# Pré-checagem dos bytes comprimidos. A validação de descompressão ocorre após definir iter_dbc.
assert arquivos, "Nenhum DBC elegível foi localizado."
primeiro_dbc = arquivos[0]["path"]
with primeiro_dbc.open("rb") as f:
    assinatura = f.read(10)
if len(assinatura) != 10:
    raise OSError(f"Arquivo DBC truncado: {primeiro_dbc} ({len(assinatura)} bytes iniciais).")
print("Pré-checagem DBC:", primeiro_dbc.name,
      "| tamanho:", primeiro_dbc.stat().st_size,
      "| cabeçalho:", assinatura.hex())


In [ ]:
def txt(v):
    if v is None: return ""
    if isinstance(v, bytes): return v.decode("latin-1", errors="replace").strip()
    if isinstance(v, float) and v.is_integer(): return str(int(v))
    return str(v).strip()

def norm_row(row):
    if hasattr(row, "_asdict"): row = row._asdict()
    if not hasattr(row, "items"):
        raise TypeError(f"Linha DBC inesperada: {type(row).__name__}")
    return {str(k).strip().upper(): txt(v) for k, v in row.items()}

def iter_dbc(path):
    """Lê registros de um DBC local usando dbc_reader; não cria DBF persistente."""
    try:
        for row in DbcReader(str(path)):
            yield norm_row(row)
    except Exception as exc:
        raise RuntimeError(f"Falha ao ler DBC {path}") from exc

# Pré-teste real antes de ler referências ou aplicar filtros.
try:
    exemplo = next(iter(iter_dbc(primeiro_dbc)))
except Exception as exc:
    traceback.print_exc()
    raise RuntimeError(
        f"Pré-teste DBC falhou para {primeiro_dbc.name}; análise interrompida. "
        "Consulte a exceção encadeada para distinguir falha de descompressão e DBF."
    ) from exc
print("Pré-teste DBC:", primeiro_dbc.name, "| campos:", len(exemplo),
      "| campos iniciais:", list(exemplo)[:8])

ALIASES = {
    "aih": ["N_AIH"], "ident": ["IDENT"], "diag": ["DIAG_PRINC"],
    "dt_inter": ["DT_INTER"], "dt_saida": ["DT_SAIDA"],
    "mun_res": ["MUNIC_RES"], "mun_mov": ["MUNIC_MOV"],
    "morte": ["MORTE"], "dias": ["DIAS_PERM"], "idade": ["IDADE"],
    "cod_idade": ["COD_IDADE"], "sexo": ["SEXO"], "raca": ["RACA_COR"],
    "proc": ["PROC_REA"], "valor": ["VAL_TOT"],
}
campo_real = {k: next((c for c in opts if c in exemplo), None) for k, opts in ALIASES.items()}
obrigatorios = set(ALIASES)  # todos os campos usados na assinatura do CSV
faltantes = sorted(k for k in obrigatorios if campo_real[k] is None)
print("Campos encontrados:", campo_real)
if faltantes: raise KeyError(f"Campos ausentes {faltantes}; disponíveis: {sorted(exemplo)}")

def get(row, conceito):
    nome = campo_real.get(conceito)
    return row.get(nome, "") if nome else ""

def data8(v):
    dig = re.sub(r"\D", "", txt(v))
    if len(dig) != 8: return ""
    try: datetime.strptime(dig, "%Y%m%d")
    except ValueError: return ""
    return dig

def code(v, largura=None):
    s = txt(v)
    if s.endswith(".0") and s[:-2].isdigit(): s = s[:-2]
    return s.zfill(largura) if largura and s.isdigit() else s

def money(v):
    s = txt(v).replace(",", ".")
    if not s: return ""
    try: return f"{Decimal(s):.2f}"
    except InvalidOperation: return s

COLS = [
    "UF", "COMPETENCIA_PROCESSAMENTO", "MUNICIPIO_RESIDENCIA", "MUNICIPIO_INTERNAÇÃO",
    "ANO_MES_INTERNACAO", "DATA_INTERNACAO", "DATA_SAIDA", "DIAG_PRINC",
    "PROCEDIMENTO_REALIZADO", "MORTE_HOSPITALAR", "DIAS_PERMANENCIA",
    "IDADE", "COD_IDADE", "SEXO", "RACA_COR", "VALOR_TOTAL",
]
def projetar(row, uf, comp):
    dt = data8(get(row, "dt_inter"))
    return dict(zip(COLS, [
        uf, comp, code(get(row, "mun_res"), 6), code(get(row, "mun_mov"), 6),
        dt[:6] if dt else "", dt, data8(get(row, "dt_saida")),
        code(get(row, "diag")).upper(), code(get(row, "proc"), 10),
        code(get(row, "morte")), code(get(row, "dias")), code(get(row, "idade")),
        code(get(row, "cod_idade")), code(get(row, "sexo")), code(get(row, "raca"), 2),
        money(get(row, "valor")),
    ]))

def sig(proj): return tuple(proj[c] for c in COLS)
UF_CODES = {
    "11":"Rondônia","12":"Acre","13":"Amazonas","14":"Roraima","15":"Pará","16":"Amapá",
    "17":"Tocantins","21":"Maranhão","22":"Piauí","23":"Ceará","24":"Rio Grande do Norte",
    "25":"Paraíba","26":"Pernambuco","27":"Alagoas","28":"Sergipe","29":"Bahia",
    "31":"Minas Gerais","32":"Espírito Santo","33":"Rio de Janeiro","35":"São Paulo",
    "41":"Paraná","42":"Santa Catarina","43":"Rio Grande do Sul","50":"Mato Grosso do Sul",
    "51":"Mato Grosso","52":"Goiás","53":"Distrito Federal",
}
UF_NAME_CODES = {v:k for k,v in UF_CODES.items()}
def uf_local(municipio):
    uf = code(municipio)[:2]
    return uf if uf in UF_CODES else ""


## 1. Ler referências e reproduzir filtro (após pré-teste DBC)

A comparação do CSV é multiconjunto: linhas repetidas são preservadas. MUNIC_MOV será usado para derivar UF do local de internação; a UF do arquivo também é mantida para detectar diferenças.


In [ ]:
def read_csv(path):
    with path.open(encoding="utf-8-sig", newline="") as f:
        return list(csv.DictReader(f, delimiter=";"))

prelim = read_csv(CSV_N18)
tabnet = read_csv(CSV_TABNET)
assert len(prelim) == 368_917
assert all(r["DIAG_PRINC"].startswith("N18") for r in prelim)
assert all("20240101" <= r["DATA_INTERNACAO"] <= "20251231" for r in prelim)
MONTHS = dict(zip("Jan Fev Mar Abr Mai Jun Jul Ago Set Out Nov Dez".split(),
                  "01 02 03 04 05 06 07 08 09 10 11 12".split()))
month_cols = [c for c in tabnet[0] if re.fullmatch(
    r"20\d\d/(Jan|Fev|Mar|Abr|Mai|Jun|Jul|Ago|Set|Out|Nov|Dez)", c)]
assert len(tabnet) == 27 and len(month_cols) == 24
print("CSV preliminar:", len(prelim), "| MORTE=1:", sum(r["MORTE_HOSPITALAR"]=="1" for r in prelim))
print("TabNet:", len(tabnet), "UFs;", len(month_cols)*len(tabnet), "células mensais")

ident_n18, por_arquivo = Counter(), Counter()
raw_local, ident1_local, raw_file = Counter(), Counter(), Counter()
ids_celula, sig_dbc, candidate_ids = defaultdict(set), Counter(), set()
counts = Counter()
t0 = time.time()
for i, info in enumerate(arquivos, 1):
    for row in iter_dbc(info["path"]):
        counts["linhas_dbc"] += 1
        diag = code(get(row, "diag")).upper()
        if not diag.startswith("N18"): continue
        counts["n18"] += 1
        dt = data8(get(row, "dt_inter"))
        if not dt:
            counts["n18_data_invalida"] += 1
            continue
        if not "20240101" <= dt <= "20251231":
            counts["n18_fora_periodo"] += 1
            continue
        counts["n18_periodo"] += 1
        ident_n18[code(get(row, "ident")) or "(vazio)"] += 1
        comp, uf_arq = info["comp"], info["uf_arq"]
        por_arquivo[(comp, uf_arq)] += 1
        sig_dbc[sig(projetar(row, uf_arq, comp))] += 1
        mes, local = dt[:6], uf_local(get(row, "mun_mov"))
        aih = code(get(row, "aih"))
        if aih: candidate_ids.add(aih)
        else: counts["n18_sem_aih"] += 1
        if not local:
            counts["municipio_local_invalido"] += 1
        else:
            raw_local[(local, mes)] += 1
            if code(get(row, "ident")) == "1": ident1_local[(local, mes)] += 1
            if aih: ids_celula[(local, mes)].add(aih)
            if local != uf_arq: counts["uf_arquivo_diferente_local"] += 1
        raw_file[(uf_arq, mes)] += 1
    if i % 50 == 0 or i == len(arquivos):
        print(f"{i}/{len(arquivos)} arquivos; {counts['linhas_dbc']:,} linhas; {time.time()-t0:.1f}s")

print("Contagens:", dict(counts))
print("IDENT N18:", dict(ident_n18), "| AIHs candidatas:", len(candidate_ids))
print("UF arquivo diferente do município local:", counts["uf_arquivo_diferente_local"])


## 2. Comparar o CSV preliminar e reconstituir a discrepância histórica

A diferença de 12 linhas histórica é entre o CSV de 368.917 e uma varredura exploratória de 368.905, cuja saída linha a linha e código não estão presentes no diretório. A comparação abaixo pode localizar diferenças do CSV contra a nova extração; se ambas coincidirem, as 12 linhas da rotina antiga não são identificáveis sem recuperar essa rotina ou sua saída.


In [ ]:
sig_prelim = Counter(sig({c:r.get(c,"").strip() for c in COLS}) for r in prelim)
only_csv, only_dbc = sig_prelim - sig_dbc, sig_dbc - sig_prelim
print("Nova extração:", counts["n18_periodo"], "| CSV:", len(prelim))
print("Só CSV:", sum(only_csv.values()), "| só DBC:", sum(only_dbc.values()))
print("Diferença frente à varredura histórica 368.905:", counts["n18_periodo"] - 368_905)

def expandir(counter, lado):
    out=[]
    for signature, n in counter.items():
        row=dict(zip(COLS, signature))
        row.update({"lado":lado,"contagem":n})
        out.append(row)
    return out

divergencias = expandir(only_csv,"somente_csv") + expandir(only_dbc,"somente_dbc")
del sig_prelim, sig_dbc
print("Assinaturas de divergência:", len(divergencias))


## 3. Examinar versões por N_AIH

Uma segunda passagem visita todos os registros dos N_AIH que apareceram em N18 no período, incluindo versões em que diagnóstico/data tenham mudado. Primeira/última competência são sensibilidades, não regra final. Empates conflitantes no mesmo mês não são desempatatados pela ordem do arquivo. IDs permanecem somente na memória.


In [ ]:
FIELDS_VERSION = ("uf_arq","ident","diag","dt_inter","dt_saida","mun_res","mun_mov","morte","proc","dias","idade","cod_idade","sexo","raca","valor")
def snap(row, uf_arq, comp):
    dt=data8(get(row,"dt_inter")); mun=code(get(row,"mun_mov"),6)
    return {
        "comp":comp,"uf_arq":uf_arq,"ident":code(get(row,"ident")),
        "diag":code(get(row,"diag")).upper(),"dt_inter":dt,"mes":dt[:6] if dt else "",
        "dt_saida":data8(get(row,"dt_saida")),"mun_res":code(get(row,"mun_res"),6),
        "mun_mov":mun,"uf":uf_local(mun),"morte":code(get(row,"morte")),
        "proc":code(get(row,"proc"),10),"dias":code(get(row,"dias")),
        "idade":code(get(row,"idade")),"cod_idade":code(get(row,"cod_idade")),
        "sexo":code(get(row,"sexo")),"raca":code(get(row,"raca"),2),
        "valor":money(get(row,"valor")),
    }
def fp(s):
    return hashlib.sha256(repr(tuple(s[k] for k in FIELDS_VERSION)).encode("utf-8")).hexdigest()

versions={}
seen=0
for i, info in enumerate(arquivos,1):
    for row in iter_dbc(info["path"]):
        key=code(get(row,"aih"))
        if not key or key not in candidate_ids: continue
        seen+=1; s=snap(row,info["uf_arq"],info["comp"]); h=fp(s)
        state=versions.get(key)
        if state is None:
            versions[key]={"first":s,"first_comp":info["comp"],"firsts":{h:s},
                           "last_comp":info["comp"],"lasts":{h:s},
                           "n":1,"comps":{info["comp"]},"payloads":{h}}
        else:
            state["n"]+=1; state["comps"].add(info["comp"]); state["payloads"].add(h)
            if info["comp"] < state["first_comp"]:
                state["first_comp"]=info["comp"]; state["firsts"]={h:s}; state["first"]=s
            elif info["comp"] == state["first_comp"]:
                state["firsts"][h]=s
            if info["comp"] > state["last_comp"]:
                state["last_comp"]=info["comp"]; state["lasts"]={h:s}
            elif info["comp"] == state["last_comp"]:
                state["lasts"][h]=s
    if i % 50 == 0 or i == len(arquivos):
        print(f"{i}/{len(arquivos)} DBCs; {seen:,} linhas de AIHs candidatas")

assert set(versions)==candidate_ids
print("AIHs:",len(versions),
      "| em várias competências:",sum(len(v["comps"])>1 for v in versions.values()),
      "| payload alterado:",sum(len(v["payloads"])>1 for v in versions.values()),
      "| empate conflitante na última:",sum(len(v["lasts"])>1 for v in versions.values()),
      "| empate conflitante na primeira:",sum(len(v["firsts"])>1 for v in versions.values()))


In [ ]:
def inside(s):
    return bool(s["dt_inter"] and "20240101" <= s["dt_inter"] <= "20251231")
def selected_n18(s):
    return s["diag"].startswith("N18") and inside(s)

scenarios=defaultdict(Counter)
for cell, ids in ids_celula.items():
    scenarios["N_AIH distinto em qualquer versão"][cell]=len(ids)

changes=Counter()
for state in versions.values():
    if len(state["firsts"])==1:
        first=next(iter(state["firsts"].values()))
        if selected_n18(first) and first["uf"]:
            scenarios["primeiro snapshot N_AIH"][(first["uf"],first["mes"])]+=1
    else:
        changes["empate conflitante na primeira competência"]+=1
    if len(state["lasts"])!=1:
        changes["empate conflitante na última competência"]+=1
        continue
    last=next(iter(state["lasts"].values()))
    if selected_n18(last) and last["uf"]:
        cell=(last["uf"],last["mes"])
        scenarios["último snapshot N_AIH"][cell]+=1
        if last["ident"]=="1": scenarios["último snapshot IDENT=1"][cell]+=1
        if last["ident"]!="5": scenarios["último snapshot IDENT diferente de 5"][cell]+=1
    if len(state["firsts"])==1:
        first=next(iter(state["firsts"].values()))
        if first["diag"].startswith("N18") != last["diag"].startswith("N18"): changes["mudou diagnóstico N18"]+=1
        if inside(first) != inside(last): changes["mudou status da data"]+=1
        if first["ident"] != last["ident"]: changes["mudou IDENT"]+=1
        if first["uf"] != last["uf"]: changes["mudou UF local"]+=1
        for field in FIELDS_VERSION:
            if first[field] != last[field]: changes["mudou "+field]+=1

scenarios["linhas N18 por UF local/mês"]=Counter(raw_local)
scenarios["linhas N18 IDENT=1 por UF local/mês"]=Counter(ident1_local)
print("Mudanças primeira/última versão:",dict(changes))


## 4. Comparação TabNet célula a célula

A UF de internação vem do prefixo de MUNIC_MOV. Célula TabNet vazia ou hífen permanece ausente; não é transformada em zero. N18 é comparado a N17–N19 nas células mensais observadas.


In [ ]:
tab_cells={}
missing_cells=[]
for row in tabnet:
    uf=UF_NAME_CODES.get(row["UF"].strip())
    if not uf: raise ValueError(f"UF desconhecida: {row['UF']}")
    for label in month_cols:
        year,month=label.split("/")
        cell=(uf,year+MONTHS[month])
        value=row[label].strip()
        if value in ("","-"): missing_cells.append(cell)
        else: tab_cells[cell]=int(value)

def compare(name, counts_by_cell):
    pairs=[(counts_by_cell.get(k,0),v) for k,v in tab_cells.items()]
    errors=[abs(a-b) for a,b in pairs]
    return {
        "cenario":name,"total_n18_celulas_observadas":sum(a for a,b in pairs),
        "total_tabnet_n17_n19":sum(b for a,b in pairs),
        "celulas":len(pairs),"n18_maior":sum(a>b for a,b in pairs),
        "iguais":sum(a==b for a,b in pairs),
        "desvio_abs_medio":round(sum(errors)/len(errors),2) if errors else None,
        "desvio_abs_total":sum(errors),
    }

summary=[compare(name,c) for name,c in scenarios.items()]
print("Células TabNet ausentes:",missing_cells)
for row in summary: print(row)

excesses=[]
for name,counts_by_cell in scenarios.items():
    for cell,broad in tab_cells.items():
        n18=counts_by_cell.get(cell,0)
        if n18>broad:
            uf,month=cell
            excesses.append({"cenario":name,"uf":UF_CODES[uf],"codigo_uf":uf,
                             "competencia":month,"n18":n18,"n17_n19":broad,"excesso":n18-broad})
for name in scenarios:
    rows=[r for r in excesses if r["cenario"]==name]
    print(name,":",len(rows),"células excedem categoria ampla")
    for r in sorted(rows,key=lambda r:r["excesso"],reverse=True)[:8]: print(r)


## 5. Interpretação e exportação

Não declare reconciliação apenas por proximidade do total nacional. Examine contagens por arquivo, linhas divergentes, mudanças de diagnóstico/data/IDENT/local, definição publicada de “Internações” e células mensais. A janela dos DBCs encerra em processamento 2025-12; a série TabNet pode estar mais atualizada.

AIH não equivale a paciente, e a exclusão de prorrogações não elimina transferências/reinternações. Se os filtros/unidade continuarem incompatíveis, mantenha SIH preliminar. A análise é descritiva e não causal.


In [ ]:
OUT = DERIVADOS / "auditoria_sih_dbc"
OUT.mkdir(exist_ok=True)
ARQUIVOS_SAIDA = [
    "resumo.csv", "contagens_por_dbc.csv", "sensibilidade_tabnet.csv",
    "celulas_acima_tabnet.csv", "divergencias_csv_vs_dbc.csv",
]
existentes = [name for name in ARQUIVOS_SAIDA if (OUT / name).exists()]
if existentes:
    raise FileExistsError(
        "Saídas existentes preservadas; mova/renomeie-as antes de executar: "
        + ", ".join(existentes)
    )

def escrever_csv(path, rows, columns):
    with path.open("w",encoding="utf-8-sig",newline="") as f:
        w=csv.DictWriter(f,fieldnames=columns,delimiter=";",extrasaction="ignore")
        w.writeheader(); w.writerows(rows)

with TemporaryDirectory(prefix="auditoria_sih_saida_", dir=DERIVADOS) as _staging:
    STAGING = Path(_staging)
    def write_new(name, rows, columns):
        escrever_csv(STAGING / name, rows, columns)

    version_summary=[
        {"indicador":"linhas_n18_reextraidas_periodo","valor":counts["n18_periodo"]},
        {"indicador":"linhas_csv_preliminar","valor":len(prelim)},
        {"indicador":"n18_fora_periodo","valor":counts["n18_fora_periodo"]},
        {"indicador":"n18_data_invalida","valor":counts["n18_data_invalida"]},
        {"indicador":"aih_candidatas","valor":len(versions)},
        {"indicador":"aih_multicompetencia","valor":sum(len(v["comps"])>1 for v in versions.values())},
        {"indicador":"aih_payload_alterado","valor":sum(len(v["payloads"])>1 for v in versions.values())},
        {"indicador":"aih_empate_ultima","valor":sum(len(v["lasts"])>1 for v in versions.values())},
        {"indicador":"linhas_so_csv","valor":sum(only_csv.values())},
        {"indicador":"linhas_so_dbc","valor":sum(only_dbc.values())},
        {"indicador":"celulas_tabnet_ausentes","valor":len(missing_cells)},
    ]
    write_new("resumo.csv",version_summary,["indicador","valor"])
    contagens_arquivo = [
        {"competencia_processamento": info["comp"], "uf_arquivo": info["uf_arq"],
         "linhas_n18_periodo": por_arquivo.get((info["comp"], info["uf_arq"]), 0)}
        for info in arquivos
    ]
    write_new("contagens_por_dbc.csv",contagens_arquivo,
              ["competencia_processamento","uf_arquivo","linhas_n18_periodo"])
    write_new("sensibilidade_tabnet.csv",summary,list(summary[0]))
    write_new("celulas_acima_tabnet.csv",excesses,["cenario","uf","codigo_uf","competencia","n18","n17_n19","excesso"])
    write_new("divergencias_csv_vs_dbc.csv",divergencias,[*COLS,"lado","contagem"])
    for name in ARQUIVOS_SAIDA:
        (STAGING / name).replace(OUT / name)
print("Saídas novas em:",OUT)
